In [1]:
"""
01_clean_with_checks.py
정제 단계: 비영어권 TV만 추리고, 품질 점검 결과를 출력해서 '무엇을 확인했고 왜 그렇게 처리했는지'를 남긴다.
(이 출력이 REPORT의 '데이터 설명: 결측치·이상치 처리 기준' 근거가 된다)
"""
import os

import numpy as np
import pandas as pd

# ===== 설정 =====
RAW_PATH = '../data/raw/all-weeks-global.tsv'
INTERIM_PATH = '../data/interim/non_english_tv_clean.csv'
CATEGORY = 'TV (Non-English)'
REQUIRED_COLS = ['week', 'category', 'weekly_rank', 'show_title', 'weekly_hours_viewed', 'cumulative_weeks_in_top_10']

# 1. 원본 데이터 로드
df_raw = pd.read_csv(RAW_PATH, sep='\t')
print(f"1. 원본 전체 데이터 크기: {df_raw.shape[0]:,}행, {df_raw.shape[1]}개 열")
print("   카테고리 종류:", sorted(df_raw['category'].dropna().unique()))
missing_cols = [c for c in REQUIRED_COLS if c not in df_raw.columns]
assert not missing_cols, f"필요한 컬럼이 없습니다: {missing_cols}"

# 2. 비영어 TV 카테고리 필터링 및 날짜 변환
df = df_raw[df_raw['category'] == CATEGORY].copy()
df['week'] = pd.to_datetime(df['week'])
df = df.sort_values(by=['week', 'weekly_rank']).reset_index(drop=True)

n_weeks = df['week'].nunique()
print(f"2. '{CATEGORY}' 필터링 완료: 총 {len(df):,}개 행")
print(f"3. 수집 주차(Week) 수: 총 {n_weeks}주")
print(f"4. 데이터 기간: {df['week'].min().strftime('%Y년 %m월 %d일')} ~ {df['week'].max().strftime('%Y년 %m월 %d일')} "
      f"(약 {(df['week'].max() - df['week'].min()).days / 365.25:.1f}년)")

# 3. 품질 점검 (처리 전에 확인하고, 문제가 있으면 눈에 띄게 알린다)
print("\n===== 데이터 품질 점검 =====")

# 3-1. 결측치
check_cols = ['week', 'weekly_rank', 'show_title', 'weekly_hours_viewed', 'cumulative_weeks_in_top_10']
na_counts = df[check_cols].isna().sum()
n_na = int(na_counts.sum())
if n_na == 0:
    print(f"[결측치] 0개 (확인한 컬럼 {len(check_cols)}개: {', '.join(check_cols)})")
else:
    print("[결측치] 컬럼별 결측 수:")
    print(na_counts.to_string())

# 3-2. 중복
n_dup_rows = int(df.duplicated().sum())
n_dup_keys = int(df.duplicated(subset=['week', 'weekly_rank']).sum())
print(f"[중복] 완전 중복 행: {n_dup_rows}개 | 같은 주·같은 순위가 두 번 나온 행: {n_dup_keys}개")

# 3-3. 주당 행 수 (Top 10이므로 주마다 10행이어야 함)
per_week = df.groupby('week').size()
off_weeks = per_week[per_week != 10]
print(f"[구조] 주당 행 수 분포: {per_week.value_counts().to_dict()} | 10행이 아닌 주: {len(off_weeks)}개")

# 3-4. 주 간격 (7일이 아닌 곳은 주가 빠진 것)
gaps = df['week'].drop_duplicates().sort_values().diff().dt.days.dropna()
n_gap = int((gaps != 7).sum())
print(f"[구조] 7일 간격이 아닌 구간: {n_gap}곳")

# 3-5. 시청 시간 값 범위
hours = df['weekly_hours_viewed']
n_nonpos = int((hours <= 0).sum())
print(f"[값] 주간 시청 시간 최소 {hours.min():,.0f} / 중앙값 {hours.median():,.0f} / 최대 {hours.max():,.0f} | 0 이하: {n_nonpos}개")

# 3-6. 이상치 확인 (주간 합계 기준, 사분위 범위 방식). 삭제 여부는 아래에서 판단
weekly_total = df.groupby('week')['weekly_hours_viewed'].sum()
q1, q3 = weekly_total.quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
outlier_weeks = weekly_total[weekly_total > fence]
top_show = df.loc[df.groupby('week')['weekly_hours_viewed'].idxmax()].set_index('week')['show_title']
print(f"[이상치] 주간 합계 상한선(Q3+1.5*IQR) {fence / 1e6:,.0f}M 초과 주: {len(outlier_weeks)}개")
for d, v in outlier_weeks.sort_values(ascending=False).head(3).items():
    print(f"   {d.date()}  {v / 1e6:,.0f}M  1위 작품: {top_show[d]}")
show_counts = top_show[outlier_weeks.index].value_counts().head(8)
print(f"[이상치] {len(outlier_weeks)}개 주의 1위 작품 분포: " + ", ".join(f"{t} {c}주" for t, c in show_counts.items()))

# 4. 처리 기준 (확인 결과를 근거로 한 판단을 기록)
print("\n===== 처리 기준 =====")
problems = []
if n_na or n_dup_rows or n_dup_keys or len(off_weeks) or n_gap or n_nonpos:
    problems.append("위 점검에서 0이 아닌 항목이 있음 -> 원인 확인 필요")
if problems:
    print("주의:", "; ".join(problems))
else:
    print("- 결측치, 중복, 주 누락, 0 이하 값이 모두 없어 행을 삭제하거나 채운 것이 없음")
print("- Top 10에 들지 못한 주는 데이터에 행이 없음: 0으로 채우지 않고 '값 없음'으로 취급 (실제 시청이 0이라는 뜻이 아니므로)")
print(f"- 이상치로 보이는 {len(outlier_weeks)}개 주는 삭제하지 않음: 오류가 아니라 실제 메가히트 사건이고, 이 분석의 핵심 대상이기 때문")
print("  (대신 평균과 중앙값을 함께 보고하고, 극단값 영향이 큰 지표는 중앙값 기준으로 해석)")

# 5. 1차 정제본 저장 (interim)
os.makedirs(os.path.dirname(INTERIM_PATH), exist_ok=True)
df.to_csv(INTERIM_PATH, index=False)
print(f"\n5. 1차 정제 파일 저장 완료 -> {INTERIM_PATH}")

# 6. 상위 3개 행 미리보기 (한글 컬럼명 및 시청 시간 단위 포맷 적용)
preview_df = df[['week', 'weekly_rank', 'show_title', 'weekly_hours_viewed', 'cumulative_weeks_in_top_10']].head(3).copy()
preview_df['week'] = preview_df['week'].dt.strftime('%Y-%m-%d')
preview_df['weekly_hours_viewed'] = preview_df['weekly_hours_viewed'].apply(lambda x: f"{x:,.0f} 시간")
preview_df = preview_df.rename(columns={
    'week': '집계 주차',
    'weekly_rank': '주간 순위',
    'show_title': '작품명',
    'weekly_hours_viewed': '주간 시청 시간',
    'cumulative_weeks_in_top_10': 'Top 10 누적 진입(주)'
})

preview_df

1. 원본 전체 데이터 크기: 10,960행, 9개 열
   카테고리 종류: ['Films (English)', 'Films (Non-English)', 'TV (English)', 'TV (Non-English)']
2. 'TV (Non-English)' 필터링 완료: 총 2,740개 행
3. 수집 주차(Week) 수: 총 274주
4. 데이터 기간: 2021년 07월 04일 ~ 2026년 09월 27일 (약 5.2년)

===== 데이터 품질 점검 =====
[결측치] 0개 (확인한 컬럼 5개: week, weekly_rank, show_title, weekly_hours_viewed, cumulative_weeks_in_top_10)
[중복] 완전 중복 행: 0개 | 같은 주·같은 순위가 두 번 나온 행: 0개
[구조] 주당 행 수 분포: {10: 274} | 10행이 아닌 주: 0개
[구조] 7일 간격이 아닌 구간: 0곳
[값] 주간 시청 시간 최소 800,000 / 중앙값 13,210,000 / 최대 571,760,000 | 0 이하: 0개
[이상치] 주간 합계 상한선(Q3+1.5*IQR) 313M 초과 주: 22개
   2021-10-03  755M  1위 작품: Squid Game
   2024-12-29  697M  1위 작품: Squid Game
   2025-01-05  661M  1위 작품: Squid Game
[이상치] 22개 주의 1위 작품 분포: Squid Game 10주, All of Us Are Dead 3주, Money Heist 2주, Bon Appétit, Your Majesty 2주, Teach You a Lesson 2주, The Glory 1주, Berlin and the Jewels of Paris 1주, Agent Kim Reactivated 1주

===== 처리 기준 =====
- 결측치, 중복, 주 누락, 0 이하 값이 모두 없어 행을 삭제하거나 채운 것이 없음
- Top 10에 들지 못한 주는 데이터에 행이 없

,집계 주차,주간 순위,작품명,주간 시청 시간,Top 10 누적 진입(주)
0,2021-07-04,1,Elite,"38,720,000 시간",1
1,2021-07-04,2,The Seven Deadly Sins,"33,240,000 시간",1
2,2021-07-04,3,"Newly Rich, Newly Poor","28,270,000 시간",1


In [1]:
import pandas as pd

raw = pd.read_csv("../data/raw/all-weeks-global.tsv", sep="\t")
print("원본 전체 행 수:", len(raw))
print("컬럼:", list(raw.columns))

원본 전체 행 수: 10960
컬럼: ['week', 'category', 'weekly_rank', 'show_title', 'season_title', 'weekly_hours_viewed', 'runtime', 'weekly_views', 'cumulative_weeks_in_top_10']


In [3]:
print(raw["category"].value_counts())
print()
print("주 수:", raw["week"].nunique())
print("값이 비어 있는 개수:")
print(raw[["weekly_hours_viewed", "runtime", "weekly_views"]].isna().sum())

# weekly_views가 처음 값이 있는 주
print("weekly_views 첫 값 주:", raw.loc[raw["weekly_views"].notna(), "week"].min())

# 정제 데이터 행 수
clean = pd.read_csv("../data/interim/non_english_tv_clean.csv")
print("정제 데이터 행 수:", len(clean))

category
Films (English)        2740
Films (Non-English)    2740
TV (English)           2740
TV (Non-English)       2740
Name: count, dtype: int64

주 수: 274
값이 비어 있는 개수:
weekly_hours_viewed       0
runtime                4080
weekly_views           4080
dtype: int64
weekly_views 첫 값 주: 2023-06-18
정제 데이터 행 수: 2740


In [4]:
w = pd.to_datetime(sorted(clean["week"].unique()))
print("주 수:", len(w), "| 모두 7일 간격:", (pd.Series(w).diff().dropna() == pd.Timedelta(days=7)).all())

주 수: 274 | 모두 7일 간격: True
